## Ví dụ cho 10.3 

Khởi tạo dữ liệu 

In [74]:
import numpy as np 
import matplotlib.pyplot as plt 
from scipy.spatial.distance import cdist 
import random 

np.random.seed(18)
means = [[2,2],[8,3],[3,6]]
cov = [[1,0],[0,1]] #ma trận hiệp phương sai (ma trận đơn vị là 1 TH của ma trận hiệp phương sai) 
N = 500 

#Tạo dữ liệu = lấy ngẫu nhiên 500 điểm cho mỗi cụm theo phân phối chuẩn có kỳ vọng
# là các mean vector ( cũng chính là các cụm) và ma trận hiệp phương sai. 
X0 = np.random.multivariate_normal(means[0],cov,N)
X1 = np.random.multivariate_normal(means[1],cov,N)
X2 = np.random.multivariate_normal(means[2],cov,N)

X = np.concatenate((X0,X1,X2), axis = 0)
K = 3 # 3 clusters
original_label = np.asarray([0]*N +[1]*N + [2]*N) 

Khai báo các hàm cần thiết 

In [75]:
def kmeans_init_centroids(X,k):
    # random pick k rows of X as initial centroids
    return X[np.random.choice(X.shape[0], k, replace = False)]

def kmeans_assign_labels(X,centroids):
    #calculate pairwise distance btw data and centroids 
    D = cdist(X, centroids) #Nếu X(1500,2), centroids(3,2) thì D(1500,3) => Mỗi hàng = 1 điểm dữ liệu, mỗi cột là một 1 centroid 
    return np.argmin(D,axis = 1) # Lấy chỉ số của giá trị nhỏ nhất trên mỗi hàng. Nếu hàng 0,cột 0 nhỏ nhất thì kết quả là 0 
    # kết quả trả về là dạng vector [0,2,1,0,...]

def has_converged(centroids, new_centroids):
    #return True if two sets of centroids are the same
    return (set([tuple(a) for a in centroids])) == set([tuple(a) for a in new_centroids])
    # Chuyển về dạng set bởi vì là set không cho phép phần tử list, nhưng cho phép tuple
    
def kmeans_update_centroids(X, labels, K):
    centroids = np.zeros((K,X.shape[1]))
    for k in range(K):
        #collect all points that are assigned to the k-th cluster 
        Xk = X[labels == k, :] # Lấy tất cả các dòng trong X mà nhãn = k, kết quả trả về là ma trận  
        centroids[k,:] = np.mean(Xk,axis = 0) # tính trung bình theo cột, ghi vào dòng tương ứng trong ma trận centroids
    return centroids # trả về ma trận (k,2)


Phần chính của phân cụm K-means

In [76]:
def kmeans(X,K):
    centroids = [kmeans_init_centroids(X,K)]
    labels = []
    it = 0 
    while True:
        labels.append(kmeans_assign_labels(X,centroids[-1]))
        new_centroids = kmeans_update_centroids(X,labels[-1],K)
        if has_converged(centroids[-1],new_centroids):
            break 
        centroids.append(new_centroids)
        it += 1
    return (centroids, labels, it)

Áp dụng thuật toán vừa viết cho dữ liệu ban đầu 

In [77]:
centroids, labels, it = kmeans(X,K)
print("Centers found by our algorithm:\n", centroids[-1])
print("Number of loop:",it)
print("Labels from our algorithm:", labels[-1][-5:])
print("Original labels:", original_label[-5:])

Centers found by our algorithm:
 [[3.02702878 5.95686115]
 [8.07476866 3.01494931]
 [1.9834967  1.96588127]]
Number of loop: 7
Labels from our algorithm: [2 0 0 0 0]
Original labels: [2 2 2 2 2]


Kết quả kiểm nghiệm bằng thư viện skicit-learn

In [78]:
from sklearn.cluster import KMeans
model = KMeans(n_clusters = 3, random_state = 0).fit(X)
print("Center found by skicit-leanr:")
print(model.cluster_centers_)
pred_label = model.predict(X)

Center found by skicit-leanr:
[[8.07476866 3.01494931]
 [3.02521978 5.94885115]
 [1.98112961 1.95794411]]


## Phân cụm chữ số viết tay 

Tải về MNIST 

In [79]:
import numpy as np 
from sklearn.datasets import fetch_openml

data_dir = '../../data' # path to your data folder 
mnist = fetch_openml("mnist_784", data_home = data_dir, as_frame = False) # as_frame trả về dữ liệu là numpy 
print("Shape of mnist data:", mnist.data.shape)


Shape of mnist data: (70000, 784)


Có 70000 mẫu, mỗi mẫu có kích thước 784. Các điểm dữ liệu được lưu dưới dạng vector hàng, với mỗi cột trong hàng đó là một số tự nhiên 0-255 đại diện cho giá trị của điểm ảnh (vector hàng này được tạo thành bằng việc chồng các cột của ma trận điểm ảnh lên thành vector cột, rồi tranpose)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors

K = 10 # number of clusters 
N = 10000
X = mnist.data[np.random.choice(mnist.data.shape[0], N)]
kmeans = KMeans(n_clusters=K).fit(X) #fit(X) là áp dụng thuật toán cho dữ liệu X 
pred_label = kmeans.predict(X) #gán nhãn cụm cho các điểm mới bằng tìm centroid gần nhất 
# kết quả trả về là vector 1 chiều dạng [0,1,2,1...]

[ 0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -6.50521303e-19 -1.38777878e-17 -1.38777878e-17 -7.58941521e-18
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -6.93889390e-18  2.42861287e-17 -6.24500451e-17 -2.22044605e-16
  1.45716772e-16  2.77555756e-17  2.84494650e-16 -2.22044605e-16
 -1.30451205e-15  1.13797860e-15 -8.88178420e-16  1.43698469e-01
  2.67373380e-01  7.06713781e-03 -3.46944695e-17 -9.02056208e-17
  6.93889390e-17 -3.46944695e-18  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
  0.00000000e+00  0.00000000e+00  0.00000000e+00  0.00000000e+00
 -1.64798730e-17  3.98986